# 광고 A/B 테스트 분석

## 분석 목적
광고 그룹(ad)과 공익광고 그룹(psa)의 사용자 전환율이 다른지 확인한다.
그룹의 의미는 데이터 설명서의 정의를 따른다.

## 주요 지표
전환율 = 전환한 사용자 수 / 전체 사용자 수

## 가설
- 귀무가설 H₀: 두 그룹의 모집단 전환율은 같다.
- 대립가설 H₁: 두 그룹의 모집단 전환율은 다르다.

## 검정 계획
- 분석 단위: 사용자
- 주요 검정: 독립된 두 집단의 비율 차이에 대한 z검정
- 유의수준: 0.05
- 검정 방향: 양측
- 효과 크기: ad 전환율 − psa 전환율
- 불확실성: 전환율 차이의 95% 신뢰구간

## 분석의 한계
계획 배정 비율을 확인할 수 없어 SRM은 평가하지 않는다.
관측 비율을 계획 비율로 대신 사용하지 않는다.

무작위 배정 과정도 확인되지 않았으므로 탐색적 비교로 진행한다.
검정 결과는 사용자 간 독립성 등 통계적 전제를 가정한 결과이며,
관측 차이를 광고의 인과 효과로 단정하지 않는다.

이미 결과를 확인한 상태에서 작성한 계획은 사전 분석 계획으로
표현하지 않는다.

## 데이터 불러오기

pandas는 데이터 처리, scipy는 통계 검정에 사용한다.
사용자 ID는 식별자이므로 문자열로 읽어 원래 표기를 보존한다.

원본 데이터는 별도로 유지하고 복사본을 분석한다.

from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display, Markdown

CSV_PATH = Path("/Users/limsunghun/Downloads/marketing_AB.csv")

ALPHA = 0.05
GROUPS = ["ad", "psa"]

# 계획 비율을 알 수 없으므로 설정하지 않습니다.
EXPECTED_AD_RATIO = None

raw_df = pd.read_csv(
    CSV_PATH,
    dtype={"user id": "string"},
)

df = raw_df.copy()
rows_before = len(raw_df)
processing_notes = []

print(f"불러온 행 수: {len(df):,}")
display(df.head())

## 기본 데이터 확인

행 수, 컬럼, 자료형과 결측치를 확인한다.
결측치가 발견돼도 즉시 삭제하지 않고 발생 위치와 원인을 조사한다.

빈 문자열이나 공백만 있는 문자열은 별도로 집계한다.

In [85]:
print(f"행 수: {df.shape[0]:,}")
print(f"컬럼 수: {df.shape[1]:,}")
print("컬럼:", df.columns.tolist())

blank_counts = pd.Series(0, index=df.columns, dtype="int64")

for column in df.select_dtypes(include=["object", "string"]).columns:
    blank_counts[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .eq("")
        .fillna(False)
        .sum()
    )

basic_summary = pd.DataFrame({
    "자료형": df.dtypes.astype(str),
    "결측치 수": df.isna().sum(),
    "결측률(%)": (df.isna().mean() * 100).round(3),
    "공백 문자열 수": blank_counts,
})

display(basic_summary)

required_columns = {"user id", "test group", "converted"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"필수 컬럼이 없습니다: {sorted(missing_columns)}")

if df.empty:
    raise ValueError("분석할 데이터가 없습니다.")

행 수: 588,101
컬럼 수: 7
컬럼: ['Unnamed: 0', 'user id', 'test group', 'converted', 'total ads', 'most ads day', 'most ads hour']


,자료형,결측치 수,결측률(%),공백 문자열 수
Unnamed: 0,int64,0,0.0,0
user id,string,0,0.0,0
test group,str,0,0.0,0
converted,bool,0,0.0,0
total ads,int64,0,0.0,0
most ads day,str,0,0.0,0
most ads hour,int64,0,0.0,0


## 중복 및 배정 일관성 확인

다음 항목을 구분해서 확인한다.

- 사용자 ID 누락
- 동일 사용자 ID의 반복 등장
- 같은 사용자의 여러 처치 라벨 등장
- 허용되지 않은 처치 라벨
- 내용 전체가 동일한 중복 기록

중복 사용자가 발견되면 임의로 첫 행만 남기지 않는다.
반복 노출 기록인지, 수집 오류인지 확인한 후 처리 기준을 정해야 한다.

저장 인덱스로 보이는 Unnamed 컬럼은 완전 중복 비교에서만 제외한다.

In [86]:
# 허용하는 실험 그룹 라벨
GROUPS = ["ad", "psa"]

ids = df["user id"]

missing_id = (
    ids.isna()
    | ids.str.strip().eq("").fillna(False)
)

duplicate_id = (
    ~missing_id
    & ids.duplicated(keep=False)
)

extra_duplicate_id = (
    ~missing_id
    & ids.duplicated(keep="first")
)

group_count_per_user = (
    df.loc[~missing_id]
    .groupby("user id")["test group"]
    .nunique(dropna=True)
)

conflicting_ids = group_count_per_user[
    group_count_per_user > 1
].index

invalid_group = ~df["test group"].isin(GROUPS)

index_columns = [
    column for column in df.columns
    if column.startswith("Unnamed:")
]

content_columns = [
    column for column in df.columns
    if column not in index_columns
]

exact_duplicates = df.duplicated(
    subset=content_columns,
    keep="first",
)

quality_summary = pd.Series({
    "누락·공백 ID 행 수": int(missing_id.sum()),
    "중복 사용자 수": int(ids[duplicate_id].nunique()),
    "중복 사용자에 해당하는 전체 행 수": int(duplicate_id.sum()),
    "사용자별 첫 행 이후 추가 행 수": int(extra_duplicate_id.sum()),
    "여러 처치 라벨에 등장한 사용자 수": len(conflicting_ids),
    "누락·잘못된 처치 라벨 행 수": int(invalid_group.sum()),
    "내용이 완전히 동일한 추가 행 수": int(exact_duplicates.sum()),
})

display(quality_summary.to_frame("건수"))

print("\n[처치 라벨별 행 수]")
display(df["test group"].value_counts(dropna=False).to_frame("행 수"))

if duplicate_id.any():
    print("\n[중복 사용자 예시]")
    display(
        df.loc[duplicate_id]
        .sort_values("user id")
        .head(10)
    )

if len(conflicting_ids) > 0:
    print("\n[배정 충돌 예시]")
    display(
        df.loc[df["user id"].isin(conflicting_ids)]
        .sort_values("user id")
        .head(10)
    )

print(
    "\n현재 파일의 test group만으로는 "
    "배정 라벨과 실제 광고 노출이 일치했는지 확인할 수 없습니다."
)

,건수
누락·공백 ID 행 수,0
중복 사용자 수,0
중복 사용자에 해당하는 전체 행 수,0
사용자별 첫 행 이후 추가 행 수,0
여러 처치 라벨에 등장한 사용자 수,0
누락·잘못된 처치 라벨 행 수,0
내용이 완전히 동일한 추가 행 수,0



[처치 라벨별 행 수]


,행 수
test group,
ad,564577
psa,23524



현재 파일의 test group만으로는 배정 라벨과 실제 광고 노출이 일치했는지 확인할 수 없습니다.


## 전환 값 확인 및 처리 기록

converted는 True/False 또는 1/0이어야 한다.

사용자 ID 누락, 중복, 잘못된 처치 라벨 또는 잘못된 전환 값이 있으면
사용자 단위 검정의 기준이 불명확하므로 실행을 중단한다.

문제가 없으면 모든 행을 유지하고 전환 값만 계산용 0/1로 변환한다.

In [87]:
print("[converted 값 분포]")
display(df["converted"].value_counts(dropna=False).to_frame("행 수"))

invalid_conversion = ~df["converted"].isin([True, False, 1, 0])

blocking_reasons = []

if missing_id.any():
    blocking_reasons.append("누락·공백 사용자 ID")

if duplicate_id.any():
    blocking_reasons.append("중복 사용자 ID")

if invalid_group.any():
    blocking_reasons.append("누락·잘못된 처치 라벨")

if invalid_conversion.any():
    blocking_reasons.append("누락·잘못된 전환 값")

if set(df["test group"].dropna().unique()) != set(GROUPS):
    blocking_reasons.append("ad 또는 psa 그룹 부재")

if blocking_reasons:
    raise ValueError(
        "처리 기준을 먼저 확인하세요: " + ", ".join(blocking_reasons)
    )

analysis_df = df.copy()
analysis_df["converted"] = analysis_df["converted"].astype(int)

processing_notes = [
    "기본 점검상 사용자 단위 분석을 막는 문제가 없어 모든 행을 유지함.",
    "converted를 계산용 0/1로 변환함.",
    "Unnamed 컬럼은 완전 중복 비교에서만 제외하고 원본에는 유지함.",
]

rows_after = len(analysis_df)

print(f"처리 전: {rows_before:,}행")
print(f"처리 후: {rows_after:,}행")
print(f"제거: {rows_before - rows_after:,}행")

for note in processing_notes:
    print("-", note)

[converted 값 분포]


,행 수
converted,
False,573258
True,14843


처리 전: 588,101행
처리 후: 588,101행
제거: 0행
- 기본 점검상 사용자 단위 분석을 막는 문제가 없어 모든 행을 유지함.
- converted를 계산용 0/1로 변환함.
- Unnamed 컬럼은 완전 중복 비교에서만 제외하고 원본에는 유지함.


## 배정 비율과 SRM

실제 그룹별 사용자 수와 비율을 확인한다.

SRM은 실제 그룹 크기를 사전에 계획한 배정 비율과 비교하는 검정이다.
이 데이터에서는 계획 비율을 확인하지 못했으므로 평가 불가능하다.

관측 비율이 약 96:4라는 이유로 계획 비율을 96:4로 설정하지 않는다.
SRM 평가 불가는 SRM 통과나 SRM 실패를 의미하지 않는다.

In [88]:
allocation = (
    analysis_df["test group"]
    .value_counts()
    .reindex(GROUPS)
    .rename("사용자 수")
    .to_frame()
)

allocation["관측 비율(%)"] = (
    allocation["사용자 수"] / allocation["사용자 수"].sum() * 100
)

display(allocation.round(3))

srm_status = "평가 불가: 사전에 계획한 배정 비율 미확인"
print("SRM:", srm_status)

,사용자 수,관측 비율(%)
test group,,
ad,564577,96.0
psa,23524,4.0


SRM: 평가 불가: 사전에 계획한 배정 비율 미확인


## 요일·시간대의 노출 패턴 비교

most ads day와 most ads hour의 분포를 그룹별로 비교한다.

- 귀무가설: 그룹과 해당 변수는 독립이다.
- 대립가설: 그룹에 따라 해당 변수의 분포가 다르다.

카이제곱 독립성 검정을 사용하고, 두 검정에는 Bonferroni 보정을 적용한다.
큰 표본에서는 작은 차이도 유의할 수 있으므로 Cramér's V를 함께 보고한다.

이 변수들은 실험 중 광고 노출에서 만들어진 처치 후 변수일 수 있다.
따라서 사전 공변량 균형이나 무작위 배정의 증거로 해석하지 않는다.
이 점검은 SRM을 대체하지 않는다.

In [89]:
pattern_columns = ["most ads day", "most ads hour"]
pattern_results = []

for column in pattern_columns:
    if column not in analysis_df.columns:
        print(f"{column}: 컬럼이 없어 점검 생략")
        continue

    values = (
        analysis_df[column]
        .astype("string")
        .fillna("(결측)")
        .replace(r"^\s*$", "(공백)", regex=True)
    )

    table = pd.crosstab(
        analysis_df["test group"],
        values,
    ).reindex(GROUPS, fill_value=0)

    percentages = table.div(table.sum(axis=1), axis=0) * 100

    print(f"\n[{column}: 그룹 내 비율(%)]")
    display(percentages.round(2))

    if table.shape[1] < 2:
        print("범주가 하나뿐이므로 검정을 생략합니다.")
        continue

    chi2, p, dof, expected = stats.chi2_contingency(
        table,
        correction=False,
    )

    approximation_ok = bool((expected >= 5).all())

    n = table.to_numpy().sum()
    dimension = min(table.shape[0] - 1, table.shape[1] - 1)
    cramers_v = np.sqrt(chi2 / (n * dimension))

    pattern_results.append({
        "변수": column,
        "최소 기대 빈도": expected.min(),
        "Cramér's V": cramers_v,
        "p값": p if approximation_ok else np.nan,
        "비고": (
            "근사 조건 충족"
            if approximation_ok
            else "희소 범주로 p값 해석 보류"
        ),
    })

pattern_summary = pd.DataFrame(pattern_results)

if not pattern_summary.empty:
    # 계획한 두 검정을 기준으로 보정
    pattern_summary["보정 p값"] = (
        pattern_summary["p값"] * 2
    ).clip(upper=1)

    display(pattern_summary)

print("분포 차이만으로 배정 오류 여부를 판단하지 않습니다.")


[most ads day: 그룹 내 비율(%)]


most ads day,Friday,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
test group,,,,,,,
ad,15.73,14.80,13.96,14.58,14.01,13.21,13.71
psa,16.17,14.89,12.15,13.00,16.60,12.36,14.84



[most ads hour: 그룹 내 비율(%)]


most ads hour,0,1,10,11,12,13,14,15,16,17,...,21,22,23,3,4,5,6,7,8,9
test group,,,,,,,,,,,,,,,,,,,,,
ad,0.94,0.82,6.63,7.82,8.01,8.06,7.75,7.59,6.37,5.95,...,5.12,4.52,3.46,0.46,0.12,0.13,0.35,1.09,3.01,5.28
psa,0.96,0.79,6.31,8.76,8.76,9.22,7.95,7.77,6.82,5.88,...,4.60,3.90,2.63,0.38,0.12,0.10,0.35,1.01,2.80,5.11


,변수,최소 기대 빈도,Cramér's V,p값,비고,보정 p값
0,most ads day,3099.154730,0.020016,4.849068e-48,근사 조건 충족,9.698135e-48
1,most ads hour,28.879951,0.018082,1.094574e-28,근사 조건 충족,2.189148e-28


분포 차이만으로 배정 오류 여부를 판단하지 않습니다.


## 광고 노출량 비교

total ads의 평균, 표준편차, 중앙값과 분위수를 확인한다.
일부 큰 값의 영향을 파악하기 위해 평균만 보지 않는다.

total ads는 처치 후 변수일 수 있으므로,
이를 기준으로 사용자를 제거하거나 전환율 검정을 보정하지 않는다.

이 단계에서는 분포를 기술적으로 비교하며 별도의 유의성 검정을 하지 않는다.

In [90]:
if "total ads" not in analysis_df.columns:
    print("total ads 컬럼이 없어 점검을 생략합니다.")

else:
    ads = pd.to_numeric(analysis_df["total ads"], errors="coerce")

    invalid_ads = (
        ads.isna()
        | ~np.isfinite(ads)
        | ads.lt(0)
    )

    print(f"누락·비수치·무한대·음수 노출량 행: {invalid_ads.sum():,}")

    if invalid_ads.any():
        print("노출량 요약은 보류합니다. 해당 행은 자동 삭제하지 않습니다.")

    else:
        exposure_summary = (
            analysis_df[["test group"]]
            .assign(total_ads=ads)
            .groupby("test group")["total_ads"]
            .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99])
            .reindex(GROUPS)
        )

        display(exposure_summary.round(2))

누락·비수치·무한대·음수 노출량 행: 0


,count,mean,std,min,25%,50%,75%,90%,99%,max
test group,,,,,,,,,,
ad,564577.0,24.82,43.75,1.0,4.0,13.0,27.0,56.0,201.0,2065.0
psa,23524.0,24.76,42.86,1.0,4.0,12.0,26.0,59.0,206.0,907.0


## 그룹별 전환율

그룹별 사용자 수, 전환 수, 미전환 수와 전환율을 계산한다.

그룹 크기가 달라도 두 비율 z검정을 사용할 수 있다.
인원을 맞추기 위해 광고 그룹의 데이터를 임의로 줄이지 않는다.

정규근사 적용을 위해 각 그룹의 전환·미전환 수가
각각 10 이상인지 보수적으로 확인한다.

In [91]:
summary = (
    analysis_df.groupby("test group")["converted"]
    .agg(users="size", conversions="sum")
    .reindex(GROUPS)
)

summary["non_conversions"] = (
    summary["users"] - summary["conversions"]
)

summary["conversion_rate"] = (
    summary["conversions"] / summary["users"]
)

# 표시용 복사본만 문자열로 변환합니다.
summary_display = summary.copy()

for column in ["users", "conversions", "non_conversions"]:
    summary_display[column] = summary_display[column].map(
        lambda value: f"{value:,.0f}"
    )

summary_display["conversion_rate"] = (
    summary_display["conversion_rate"]
    .map(lambda value: f"{value:.3%}")
)

display(summary_display)

if (summary[["conversions", "non_conversions"]] < 10).any().any():
    raise ValueError(
        "전환 또는 미전환 빈도가 작습니다. "
        "아래 정규근사 분석 대신 정확검정 및 적절한 구간 추정을 검토하세요."
    )

,users,conversions,non_conversions,conversion_rate
test group,,,,
ad,"564,577","14,423","550,154",2.555%
psa,"23,524",420,"23,104",1.785%


## 전환율 차이 검정

귀무가설 아래에서는 두 그룹의 전환율이 같다고 가정하므로,
합동 전환율을 사용한 두 비율 z검정을 수행한다.

전환율 차이의 신뢰구간에는 각 그룹의 전환율을 따로 사용하는
비합동 표준오차를 사용한다.

- 전환율 차이: ad − psa
- 양수: 광고 그룹의 관측 전환율이 더 높음
- 음수: 광고 그룹의 관측 전환율이 더 낮음
- p값 < 0.05: 귀무가설 기각
- p값 ≥ 0.05: 귀무가설을 기각할 증거가 부족함

절대 차이(%p)와 상대 변화율(%)은 구분한다.
예를 들어 2%에서 3%로 증가하면 절대 차이는 1%p,
상대 증가율은 50%이다.

In [92]:
n_ad = int(summary.loc["ad", "users"])
n_psa = int(summary.loc["psa", "users"])

x_ad = int(summary.loc["ad", "conversions"])
x_psa = int(summary.loc["psa", "conversions"])

p_ad = x_ad / n_ad
p_psa = x_psa / n_psa

difference = p_ad - p_psa

# 1. 합동 표준오차: 귀무가설 검정용
pooled_rate = (x_ad + x_psa) / (n_ad + n_psa)

se_pooled = np.sqrt(
    pooled_rate * (1 - pooled_rate)
    * (1 / n_ad + 1 / n_psa)
)

z_stat = difference / se_pooled
p_value = 2 * stats.norm.sf(abs(z_stat))

# 2. 비합동 표준오차: 신뢰구간 추정용
se_unpooled = np.sqrt(
    p_ad * (1 - p_ad) / n_ad
    + p_psa * (1 - p_psa) / n_psa
)

critical_value = stats.norm.ppf(1 - ALPHA / 2)

ci_low = difference - critical_value * se_unpooled
ci_high = difference + critical_value * se_unpooled

relative_change = (
    difference / p_psa if p_psa > 0 else np.nan
)

print(f"ad 전환율: {p_ad:.3%}")
print(f"psa 전환율: {p_psa:.3%}")
print(f"절대 차이(ad − psa): {difference * 100:+.3f}%p")
print(f"상대 변화율: {relative_change:+.2%}")
print(
    "차이의 95% 신뢰구간: "
    f"[{ci_low * 100:+.3f}, {ci_high * 100:+.3f}]%p"
)
print(f"z통계량: {z_stat:.4f}")
print(f"양측 p값: {p_value:.6g}")

if p_value == 0:
    print("※ 매우 작은 p값이 수치 정밀도 한계로 0으로 표시됐습니다.")

ad 전환율: 2.555%
psa 전환율: 1.785%
절대 차이(ad − psa): +0.769%p
상대 변화율: +43.09%
차이의 95% 신뢰구간: [+0.595, +0.943]%p
z통계량: 7.3701
양측 p값: 1.70528e-13


## 결과 해석 원칙

통계적 유의성, 효과 크기, 사업적 중요성을 구분한다.

p값이 작더라도 효과가 사업적으로 충분히 크다는 뜻은 아니며,
광고 비용과 기대 수익 등을 별도로 고려해야 한다.

계획 배정 비율과 배정 과정을 확인하지 못했으므로,
이번 결과를 실험의 무결성이나 광고의 인과 효과가 입증된 것으로
해석하지 않는다.

In [93]:
if p_value < ALPHA:
    conclusion = (
        "유의수준 0.05에서 귀무가설을 기각한다. "
        "독립성 등 통계적 전제가 성립한다면 "
        "두 그룹의 모집단 전환율이 다르다는 증거가 있다."
    )
else:
    conclusion = (
        "유의수준 0.05에서 귀무가설을 기각하지 못한다. "
        "두 그룹의 모집단 전환율이 다르다는 증거가 부족하다. "
        "이는 두 전환율이 같다는 증명은 아니다."
    )

p_text = (
    f"{p_value:.6g}"
    if p_value > 0
    else "수치 정밀도 한계로 0 표시"
)

report = f"""
# 광고 A/B 테스트 분석 결과

## 데이터 처리
- 처리 전: {rows_before:,}행
- 처리 후: {rows_after:,}행
- 제거한 행: {rows_before - rows_after:,}행
- 처리 이유: 기본 점검을 통과한 모든 사용자를 유지하고 전환 값을 0/1로 변환했다.

## 주요 결과

| 항목 | 결과 |
|---|---|
| ad 사용자 수 | {n_ad:,} |
| psa 사용자 수 | {n_psa:,} |
| ad 전환 수 | {x_ad:,} |
| psa 전환 수 | {x_psa:,} |
| ad 전환율 | {p_ad:.3%} |
| psa 전환율 | {p_psa:.3%} |
| 절대 차이(ad − psa) | {difference * 100:+.3f}%p |
| 상대 변화율 | {relative_change:+.2%} |
| 차이의 95% 신뢰구간 | [{ci_low * 100:+.3f}, {ci_high * 100:+.3f}]%p |
| 검정 | 두 비율 z검정, 양측 |
| 유의수준 | {ALPHA} |
| p값 | {p_text} |
| SRM | {srm_status} |

## 통계적 판단
{conclusion}

## 제한 사항
- 계획 배정 비율이 확인되지 않아 SRM은 평가하지 못했다.
- 사용자 단위의 독립적인 무작위 배정 과정은 확인되지 않았다.
- 요일·시간대·노출량 점검은 SRM이나 무작위 배정의 검증을 대체하지 않는다.
- 노출 관련 변수는 처치 후 변수일 수 있어 전환율 보정에 사용하지 않았다.
- 관측 차이를 광고의 인과 효과로 단정하지 않는다.
- 사업적 중요성은 효과 크기와 비용 등을 함께 고려해 판단해야 한다.
"""

display(Markdown(report))


# 광고 A/B 테스트 분석 결과

## 데이터 처리
- 처리 전: 588,101행
- 처리 후: 588,101행
- 제거한 행: 0행
- 처리 이유: 기본 점검을 통과한 모든 사용자를 유지하고 전환 값을 0/1로 변환했다.

## 주요 결과

| 항목 | 결과 |
|---|---|
| ad 사용자 수 | 564,577 |
| psa 사용자 수 | 23,524 |
| ad 전환 수 | 14,423 |
| psa 전환 수 | 420 |
| ad 전환율 | 2.555% |
| psa 전환율 | 1.785% |
| 절대 차이(ad − psa) | +0.769%p |
| 상대 변화율 | +43.09% |
| 차이의 95% 신뢰구간 | [+0.595, +0.943]%p |
| 검정 | 두 비율 z검정, 양측 |
| 유의수준 | 0.05 |
| p값 | 1.70528e-13 |
| SRM | 평가 불가: 사전에 계획한 배정 비율 미확인 |

## 통계적 판단
유의수준 0.05에서 귀무가설을 기각한다. 독립성 등 통계적 전제가 성립한다면 두 그룹의 모집단 전환율이 다르다는 증거가 있다.

## 제한 사항
- 계획 배정 비율이 확인되지 않아 SRM은 평가하지 못했다.
- 사용자 단위의 독립적인 무작위 배정 과정은 확인되지 않았다.
- 요일·시간대·노출량 점검은 SRM이나 무작위 배정의 검증을 대체하지 않는다.
- 노출 관련 변수는 처치 후 변수일 수 있어 전환율 보정에 사용하지 않았다.
- 관측 차이를 광고의 인과 효과로 단정하지 않는다.
- 사업적 중요성은 효과 크기와 비용 등을 함께 고려해 판단해야 한다.


## 전환율 차이의 검정과 효과 크기

converted는 전환 여부를 나타내는 이진 변수이므로,
독립된 두 집단의 비율 차이에 대한 양측 z검정을 사용한다.

- 귀무가설: ad와 psa의 모집단 전환율은 같다.
- 대립가설: 두 그룹의 모집단 전환율은 다르다.
- 유의수준: 0.05

결과는 다음 항목을 함께 보고한다.

1. 점추정: ad 전환율 − psa 전환율
2. 95% 신뢰구간: 전환율 차이의 추정 불확실성
3. 효과 크기:
   - 절대 차이(%p)
   - 상대 비율(ad 전환율 / psa 전환율, 배수)
   - 상대 변화율(%)

비율 자료에서는 절대 차이 자체도 효과 크기이다.
Cohen's d나 η² 대신 전환율의 절대·상대 차이를 사용한다.

검정에는 합동 표준오차를, 차이의 신뢰구간에는 비합동 표준오차를 사용한다.
이 결과는 사용자 간 독립성 등 통계적 전제를 가정한다.

In [94]:
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display, Markdown

ALPHA = 0.05
GROUPS = ["ad", "psa"]

# 전처리한 데이터가 준비됐는지 확인
if "analysis_df" not in globals():
    raise NameError("앞의 전처리 셀을 실행해 analysis_df를 먼저 만들어주세요.")

required = {"user id", "test group", "converted"}
if not required.issubset(analysis_df.columns):
    raise ValueError(f"필수 컬럼이 없습니다: {required - set(analysis_df.columns)}")

data = analysis_df.copy()

ids = data["user id"].astype("string")
if ids.isna().any() or ids.str.strip().eq("").any():
    raise ValueError("누락 또는 공백 사용자 ID가 있습니다.")

if ids.duplicated().any():
    raise ValueError("중복 사용자가 있습니다. 사용자 단위 집계를 확인하세요.")

if not data["test group"].isin(GROUPS).all():
    raise ValueError("누락되거나 허용되지 않은 처치 라벨이 있습니다.")

if not data["converted"].isin([True, False, 1, 0]).all():
    raise ValueError("converted에 결측치 또는 잘못된 값이 있습니다.")

data["converted"] = data["converted"].astype(int)

summary = (
    data.groupby("test group")["converted"]
    .agg(users="size", conversions="sum")
    .reindex(GROUPS)
)

if summary["users"].isna().any() or summary["users"].eq(0).any():
    raise ValueError("ad와 psa 두 그룹 모두에 사용자가 있어야 합니다.")

summary["non_conversions"] = summary["users"] - summary["conversions"]
summary["conversion_rate"] = summary["conversions"] / summary["users"]

if (summary[["conversions", "non_conversions"]] < 10).any().any():
    raise ValueError(
        "전환 또는 미전환 빈도가 작아 정규근사를 보류합니다. "
        "정확검정과 적절한 구간 추정을 검토하세요."
    )

n_ad = int(summary.loc["ad", "users"])
n_psa = int(summary.loc["psa", "users"])
x_ad = int(summary.loc["ad", "conversions"])
x_psa = int(summary.loc["psa", "conversions"])

p_ad = x_ad / n_ad
p_psa = x_psa / n_psa

# ① 점추정 및 절대 효과 크기
difference = p_ad - p_psa

# 두 비율 z검정: 합동 표준오차
pooled_rate = (x_ad + x_psa) / (n_ad + n_psa)
se_null = np.sqrt(
    pooled_rate * (1 - pooled_rate) * (1 / n_ad + 1 / n_psa)
)

z_stat = difference / se_null
p_value = 2 * stats.norm.sf(abs(z_stat))

# ② 차이의 95% 신뢰구간: 비합동 표준오차
critical = stats.norm.ppf(1 - ALPHA / 2)

se_difference = np.sqrt(
    p_ad * (1 - p_ad) / n_ad
    + p_psa * (1 - p_psa) / n_psa
)

ci_low = difference - critical * se_difference
ci_high = difference + critical * se_difference

# ③ 상대 효과 크기
risk_ratio = p_ad / p_psa
relative_change = risk_ratio - 1

# 상대 비율의 95% 신뢰구간: 로그 정규근사
se_log_rr = np.sqrt(
    1 / x_ad - 1 / n_ad
    + 1 / x_psa - 1 / n_psa
)

rr_ci_low = np.exp(np.log(risk_ratio) - critical * se_log_rr)
rr_ci_high = np.exp(np.log(risk_ratio) + critical * se_log_rr)

p_text = f"{p_value:.3g}" if p_value > 0 else "수치 정밀도 한계로 0 표시"

results = pd.DataFrame({
    "항목": [
        "ad 전환율",
        "psa 전환율",
        "점추정·절대 차이",
        "절대 차이의 95% 신뢰구간",
        "상대 비율(ad / psa)",
        "상대 비율의 95% 신뢰구간",
        "상대 변화율",
        "z통계량",
        "양측 p값",
    ],
    "결과": [
        f"{p_ad:.3%}",
        f"{p_psa:.3%}",
        f"{difference * 100:+.3f}%p",
        f"[{ci_low * 100:+.3f}, {ci_high * 100:+.3f}]%p",
        f"{risk_ratio:.3f}배",
        f"[{rr_ci_low:.3f}, {rr_ci_high:.3f}]배",
        f"{relative_change:+.2%}",
        f"{z_stat:.3f}",
        p_text,
    ],
})

display(results)

,항목,결과
0,ad 전환율,2.555%
1,psa 전환율,1.785%
2,점추정·절대 차이,+0.769%p
3,절대 차이의 95% 신뢰구간,"[+0.595, +0.943]%p"
4,상대 비율(ad / psa),1.431배
5,상대 비율의 95% 신뢰구간,"[1.300, 1.575]배"
6,상대 변화율,+43.09%
7,z통계량,7.370
8,양측 p값,1.71e-13


In [95]:
if p_value < ALPHA:
    statistical_conclusion = (
        "유의수준 0.05에서 귀무가설을 기각했다. "
        "통계적 전제가 성립한다면 두 그룹의 전환율이 다르다는 증거가 있다."
    )
else:
    statistical_conclusion = (
        "유의수준 0.05에서 귀무가설을 기각하지 못했다. "
        "두 그룹의 전환율이 다르다는 증거가 부족하며, "
        "두 전환율이 같다는 의미는 아니다."
    )

result_sentence = (
    f"광고 그룹의 관측 전환율은 {p_ad:.3%}, "
    f"공익광고 그룹은 {p_psa:.3%}였다. "
    f"전환율 차이(ad − psa)는 {difference * 100:+.3f}%p"
    f"(95% 신뢰구간 [{ci_low * 100:+.3f}, {ci_high * 100:+.3f}]%p)였고, "
    f"상대 비율은 {risk_ratio:.3f}배"
    f"(상대 변화율 {relative_change:+.2%})였다. "
    f"양측 두 비율 z검정 결과 z = {z_stat:.3f}, p = {p_text}였다."
)

display(Markdown(
    "### 결과\n\n"
    + result_sentence
    + "\n\n"
    + statistical_conclusion
    + "\n\n**주의:** SRM과 무작위 배정 과정을 확인하지 못했으므로 "
      "관측 차이를 광고의 인과 효과로 단정하지 않는다."
))

### 결과

광고 그룹의 관측 전환율은 2.555%, 공익광고 그룹은 1.785%였다. 전환율 차이(ad − psa)는 +0.769%p(95% 신뢰구간 [+0.595, +0.943]%p)였고, 상대 비율은 1.431배(상대 변화율 +43.09%)였다. 양측 두 비율 z검정 결과 z = 7.370, p = 1.71e-13였다.

유의수준 0.05에서 귀무가설을 기각했다. 통계적 전제가 성립한다면 두 그룹의 전환율이 다르다는 증거가 있다.

**주의:** SRM과 무작위 배정 과정을 확인하지 못했으므로 관측 차이를 광고의 인과 효과로 단정하지 않는다.

## 비용을 고려한 도입 기준

다음 값을 가정한다.

- 도입 대상 사용자 수 N
- 추가 전환 1건의 기여이익 V
- 비교 대안 대비 사용자당 추가 비용 C
- 고정 도입 비용 F

손익분기 전환율 차이는 다음과 같다.

손익분기 차이 = (N × C + F) / (N × V)

즉, 이 값보다 전환율이 많이 상승해야 가정한 비용을 회수한다.

이번 비교의 대조군은 psa이므로 비용 역시 psa 대비 추가 비용으로 정의한다.
광고 미노출과의 비교로 바꾸어 해석하지 않는다.

관측 전환율 차이가 실제 인과 효과이고 도입 대상에도 유지된다는
가정 아래에서만 예상 순이익 계산이 성립한다.
아래 계산은 실제 수익 예측이 아닌 가정에 따른 시나리오이다.

In [96]:
# 예시 가정값: 실제 의사결정 시 수정하세요.
TARGET_USERS = 100_000
VALUE_PER_CONVERSION = 20_000       # 추가 전환 1건의 기여이익, 원
EXTRA_COST_PER_USER = 100           # psa 대비 사용자당 추가 비용, 원
FIXED_COST = 1_000_000              # 고정 도입 비용, 원

if TARGET_USERS <= 0 or VALUE_PER_CONVERSION <= 0:
    raise ValueError("대상 사용자 수와 전환당 기여이익은 양수여야 합니다.")

total_extra_cost = (
    TARGET_USERS * EXTRA_COST_PER_USER + FIXED_COST
)

break_even_difference = (
    total_extra_cost / (TARGET_USERS * VALUE_PER_CONVERSION)
)

def scenario_profit(rate_difference):
    return (
        TARGET_USERS * rate_difference * VALUE_PER_CONVERSION
        - total_extra_cost
    )

profit_point = scenario_profit(difference)
profit_low = scenario_profit(ci_low)
profit_high = scenario_profit(ci_high)

economics = pd.DataFrame({
    "항목": [
        "대상 사용자 수",
        "전환당 기여이익",
        "사용자당 추가 비용",
        "고정 도입 비용",
        "총 추가 비용",
        "손익분기 전환율 차이",
        "관측 차이를 적용한 추가 전환 수",
        "관측 차이를 적용한 순이익",
        "차이의 신뢰구간 하한을 적용한 순이익",
        "차이의 신뢰구간 상한을 적용한 순이익",
    ],
    "가정 또는 계산값": [
        f"{TARGET_USERS:,}명",
        f"{VALUE_PER_CONVERSION:,.0f}원",
        f"{EXTRA_COST_PER_USER:,.0f}원",
        f"{FIXED_COST:,.0f}원",
        f"{total_extra_cost:,.0f}원",
        f"{break_even_difference * 100:.3f}%p",
        f"{TARGET_USERS * difference:+,.1f}건",
        f"{profit_point:+,.0f}원",
        f"{profit_low:+,.0f}원",
        f"{profit_high:+,.0f}원",
    ],
})

display(economics)

if ci_low > break_even_difference:
    economic_assessment = (
        "전환율 차이의 신뢰구간 하한도 손익분기 기준을 넘는다. "
        "설계 타당성과 비용 가정이 확인된다면 경제적으로 유망한 시나리오다."
    )
elif ci_high < break_even_difference:
    economic_assessment = (
        "전환율 차이의 신뢰구간 상한도 손익분기 기준보다 낮다. "
        "현재 비용 가정에서는 도입을 지지하기 어렵다."
    )
else:
    economic_assessment = (
        "손익분기 기준이 전환율 차이의 신뢰구간 안에 있다. "
        "현재 추정만으로 경제성을 확정하기 어렵다."
    )

print(economic_assessment)
print(
    "\n※ 이 범위는 비용과 전환 가치가 고정됐다는 가정하에 "
    "전환율 차이의 불확실성만 반영합니다. 실제 미래 이익의 예측구간은 아닙니다."
)

,항목,가정 또는 계산값
0,대상 사용자 수,"100,000명"
1,전환당 기여이익,"20,000원"
2,사용자당 추가 비용,100원
3,고정 도입 비용,"1,000,000원"
4,총 추가 비용,"11,000,000원"
5,손익분기 전환율 차이,0.550%p
6,관측 차이를 적용한 추가 전환 수,+769.2건
7,관측 차이를 적용한 순이익,"+4,384,906원"
8,차이의 신뢰구간 하한을 적용한 순이익,"+901,865원"
9,차이의 신뢰구간 상한을 적용한 순이익,"+7,867,948원"


전환율 차이의 신뢰구간 하한도 손익분기 기준을 넘는다. 설계 타당성과 비용 가정이 확인된다면 경제적으로 유망한 시나리오다.

※ 이 범위는 비용과 전환 가치가 고정됐다는 가정하에 전환율 차이의 불확실성만 반영합니다. 실제 미래 이익의 예측구간은 아닙니다.


In [97]:
# 앞의 전처리 기록이 없으면 원본 행 수를 임의로 추정하지 않습니다.
source_rows = globals().get("rows_before")
current_rows = len(analysis_df)

if source_rows is None:
    row_history = (
        f"현재 분석 데이터는 {current_rows:,}행이다. "
        "전처리 전 행 수와 제거 이력은 별도 기록 확인이 필요하다."
    )
else:
    row_history = (
        f"전처리 전 {source_rows:,}행, 분석 데이터 {current_rows:,}행이며 "
        f"순 행 수 차이는 {source_rows - current_rows:,}행이다."
    )

notes = globals().get("processing_notes", [])
notes_text = (
    "\n".join(f"- {note}" for note in notes)
    if notes
    else "- 별도 정제 메모가 없어 이전 전처리 기록을 확인해야 한다."
)

# 설계 정보가 미확인인 현재 상황에 대한 판단
decision = "데이터 추가 수집"

final_report = f"""
# 광고 A/B 테스트 최종 보고서

## 1. 질문과 가설

광고 그룹(ad)과 공익광고 그룹(psa)의 사용자 전환율은 다른가?

- 귀무가설 H₀: 두 그룹의 모집단 전환율은 같다.
- 대립가설 H₁: 두 그룹의 모집단 전환율은 다르다.
- 유의수준: 0.05
- 검정 방향: 양측

결과를 확인한 뒤 작성한 분석 계획이라면 사전 계획으로 간주하지 않는다.

## 2. 데이터 위생 점검

{row_history}

현재 분석 데이터에서 사용자 ID 누락·중복, 잘못된 처치 라벨,
전환 값 오류가 없는지 확인했다. 사용자 한 명을 한 분석 단위로 사용했다.

기존 처리 메모:
{notes_text}

사전에 계획한 배정 비율을 확인하지 못해 SRM은 평가하지 못했다.
관측 비율을 계획 비율로 대신 사용하지 않았다.
노출 요일·시간대·노출량의 비교는 SRM이나 무작위 배정의 검증을 대체하지 않는다.

## 3. 검정 선택 근거

전환 여부는 이진 변수이고 두 그룹의 비율을 비교하므로
독립된 두 집단의 비율 차이에 대한 z검정을 사용했다.
각 그룹의 전환·미전환 수가 정규근사에 충분한지 확인했다.

귀무가설 검정에는 합동 표준오차를,
전환율 차이의 95% 신뢰구간에는 비합동 표준오차를 사용했다.

## 4. 결과: 점추정·신뢰구간·효과 크기

| 항목 | 결과 |
|---|---:|
| ad 사용자 수 | {n_ad:,} |
| psa 사용자 수 | {n_psa:,} |
| ad 전환율 | {p_ad:.3%} |
| psa 전환율 | {p_psa:.3%} |
| 점추정 및 절대 효과 크기 | {difference * 100:+.3f}%p |
| 절대 차이의 95% 신뢰구간 | [{ci_low * 100:+.3f}, {ci_high * 100:+.3f}]%p |
| 상대 비율 | {risk_ratio:.3f}배 |
| 상대 비율의 95% 신뢰구간 | [{rr_ci_low:.3f}, {rr_ci_high:.3f}]배 |
| 상대 변화율 | {relative_change:+.2%} |
| z통계량 | {z_stat:.3f} |
| 양측 p값 | {p_text} |

{statistical_conclusion}

### 비용을 고려한 시나리오

다음은 실제 확인된 비용이 아닌 예시 가정이다.

- 도입 대상: {TARGET_USERS:,}명
- 추가 전환 1건의 기여이익: {VALUE_PER_CONVERSION:,.0f}원
- psa 대비 사용자당 추가 비용: {EXTRA_COST_PER_USER:,.0f}원
- 고정 도입 비용: {FIXED_COST:,.0f}원
- 손익분기 전환율 차이: {break_even_difference * 100:.3f}%p
- 관측 차이를 적용한 순이익: {profit_point:+,.0f}원
- 차이의 신뢰구간 양 끝을 적용한 순이익:
  {profit_low:+,.0f}원 ~ {profit_high:+,.0f}원

{economic_assessment}

이는 관측 차이를 인과 효과로 사용할 수 있고,
도입 대상에도 같은 효과가 유지된다는 조건의 계산이다.
실제 미래 이익의 예측구간은 아니다.

## 5. 의사결정

**선택: {decision}**

현재 결과만으로 출시를 확정하지 않는다.
우선 다음 설계·운영 정보를 확보한다.

1. 실험 당시 계획한 배정 비율과 무작위 배정 방식
2. 배정 로그와 실제 노출 로그
3. 수집 누락 및 사용자 포함·제외 기준
4. 전환 관찰 기간과 그룹 간 측정 기준의 일치 여부
5. 실제 추가 비용과 전환당 기여이익

설계 정보를 확보하면 SRM을 평가하고 경제성을 다시 검토한다.
기존 기록을 확보할 수 없다면 계획 비율과 분석 기준을
사전에 명시한 새로운 실험을 고려한다.

단순히 같은 방식으로 사용자 수를 더 늘리는 것만으로는
배정 과정의 불확실성이 해소되지 않는다.

## 6. 한계

- 무작위 배정과 사용자 간 독립성이 확인되지 않았다.
- 계획 배정 비율을 몰라 SRM을 평가하지 못했다.
- 따라서 관측 차이를 광고의 인과 효과로 단정할 수 없다.
- 표본 오차를 나타내는 신뢰구간은 선택 편향이나 수집 오류를 반영하지 않는다.
- 노출 관련 변수는 처치 후 변수일 수 있어 인과적 통제 변수로 사용하지 않았다.
- 비용 및 전환 가치는 예시 가정으로 실제 수익성을 확정할 수 없다.
- 상대 변화율만으로 성과를 표현하지 않고 절대 차이도 함께 보고했다.
- 이 데이터에서 관측한 결과가 다른 기간이나 대상에서도 유지되는지는 알 수 없다.
"""

display(Markdown(final_report))


# 광고 A/B 테스트 최종 보고서

## 1. 질문과 가설

광고 그룹(ad)과 공익광고 그룹(psa)의 사용자 전환율은 다른가?

- 귀무가설 H₀: 두 그룹의 모집단 전환율은 같다.
- 대립가설 H₁: 두 그룹의 모집단 전환율은 다르다.
- 유의수준: 0.05
- 검정 방향: 양측

결과를 확인한 뒤 작성한 분석 계획이라면 사전 계획으로 간주하지 않는다.

## 2. 데이터 위생 점검

전처리 전 588,101행, 분석 데이터 588,101행이며 순 행 수 차이는 0행이다.

현재 분석 데이터에서 사용자 ID 누락·중복, 잘못된 처치 라벨,
전환 값 오류가 없는지 확인했다. 사용자 한 명을 한 분석 단위로 사용했다.

기존 처리 메모:
- 기본 점검상 사용자 단위 분석을 막는 문제가 없어 모든 행을 유지함.
- converted를 계산용 0/1로 변환함.
- Unnamed 컬럼은 완전 중복 비교에서만 제외하고 원본에는 유지함.

사전에 계획한 배정 비율을 확인하지 못해 SRM은 평가하지 못했다.
관측 비율을 계획 비율로 대신 사용하지 않았다.
노출 요일·시간대·노출량의 비교는 SRM이나 무작위 배정의 검증을 대체하지 않는다.

## 3. 검정 선택 근거

전환 여부는 이진 변수이고 두 그룹의 비율을 비교하므로
독립된 두 집단의 비율 차이에 대한 z검정을 사용했다.
각 그룹의 전환·미전환 수가 정규근사에 충분한지 확인했다.

귀무가설 검정에는 합동 표준오차를,
전환율 차이의 95% 신뢰구간에는 비합동 표준오차를 사용했다.

## 4. 결과: 점추정·신뢰구간·효과 크기

| 항목 | 결과 |
|---|---:|
| ad 사용자 수 | 564,577 |
| psa 사용자 수 | 23,524 |
| ad 전환율 | 2.555% |
| psa 전환율 | 1.785% |
| 점추정 및 절대 효과 크기 | +0.769%p |
| 절대 차이의 95% 신뢰구간 | [+0.595, +0.943]%p |
| 상대 비율 | 1.431배 |
| 상대 비율의 95% 신뢰구간 | [1.300, 1.575]배 |
| 상대 변화율 | +43.09% |
| z통계량 | 7.370 |
| 양측 p값 | 1.71e-13 |

유의수준 0.05에서 귀무가설을 기각했다. 통계적 전제가 성립한다면 두 그룹의 전환율이 다르다는 증거가 있다.

### 비용을 고려한 시나리오

다음은 실제 확인된 비용이 아닌 예시 가정이다.

- 도입 대상: 100,000명
- 추가 전환 1건의 기여이익: 20,000원
- psa 대비 사용자당 추가 비용: 100원
- 고정 도입 비용: 1,000,000원
- 손익분기 전환율 차이: 0.550%p
- 관측 차이를 적용한 순이익: +4,384,906원
- 차이의 신뢰구간 양 끝을 적용한 순이익:
  +901,865원 ~ +7,867,948원

전환율 차이의 신뢰구간 하한도 손익분기 기준을 넘는다. 설계 타당성과 비용 가정이 확인된다면 경제적으로 유망한 시나리오다.

이는 관측 차이를 인과 효과로 사용할 수 있고,
도입 대상에도 같은 효과가 유지된다는 조건의 계산이다.
실제 미래 이익의 예측구간은 아니다.

## 5. 의사결정

**선택: 데이터 추가 수집**

현재 결과만으로 출시를 확정하지 않는다.
우선 다음 설계·운영 정보를 확보한다.

1. 실험 당시 계획한 배정 비율과 무작위 배정 방식
2. 배정 로그와 실제 노출 로그
3. 수집 누락 및 사용자 포함·제외 기준
4. 전환 관찰 기간과 그룹 간 측정 기준의 일치 여부
5. 실제 추가 비용과 전환당 기여이익

설계 정보를 확보하면 SRM을 평가하고 경제성을 다시 검토한다.
기존 기록을 확보할 수 없다면 계획 비율과 분석 기준을
사전에 명시한 새로운 실험을 고려한다.

단순히 같은 방식으로 사용자 수를 더 늘리는 것만으로는
배정 과정의 불확실성이 해소되지 않는다.

## 6. 한계

- 무작위 배정과 사용자 간 독립성이 확인되지 않았다.
- 계획 배정 비율을 몰라 SRM을 평가하지 못했다.
- 따라서 관측 차이를 광고의 인과 효과로 단정할 수 없다.
- 표본 오차를 나타내는 신뢰구간은 선택 편향이나 수집 오류를 반영하지 않는다.
- 노출 관련 변수는 처치 후 변수일 수 있어 인과적 통제 변수로 사용하지 않았다.
- 비용 및 전환 가치는 예시 가정으로 실제 수익성을 확정할 수 없다.
- 상대 변화율만으로 성과를 표현하지 않고 절대 차이도 함께 보고했다.
- 이 데이터에서 관측한 결과가 다른 기간이나 대상에서도 유지되는지는 알 수 없다.


In [98]:
from pathlib import Path

report_path = Path("ab_test_final_report.md")
report_path.write_text(final_report, encoding="utf-8")

print(f"저장 완료: {report_path.resolve()}")

저장 완료: /Users/limsunghun/python/DI/ab_test_final_report.md
